In [3]:
import os
import json
import requests
from PIL import Image, UnidentifiedImageError
import pandas as pd
from tqdm import tqdm
from openai import AzureOpenAI


In [4]:
client = AzureOpenAI(
  azure_endpoint = "https://rag-projects.openai.azure.com/", 
  api_key="Pam7HUoexk0iaxEYSnHmwhbsDT1YZrTH5J3FK78yJLCzmDPjSMXqJQQJ99ALACYeBjFXJ3w3AAABACOG6gre",  
  api_version="2024-02-01"
)

In [5]:
def generate_image_dalle3_hd(
    prompt: str,
    client,
    output_path: str
):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    result = client.images.generate(
        model="dall-e-3",
        prompt=prompt,
        size="1024x1024",
        quality="hd",  # HD setting
        n=1,
    )

    json_response = json.loads(result.model_dump_json())
    image_url = json_response["data"][0]["url"]
    image_bytes = requests.get(image_url).content

    with open(output_path, "wb") as f:
        f.write(image_bytes)

    return Image.open(output_path)

In [6]:
output_dir = 'dalle images/dog'
os.makedirs(output_dir, exist_ok=True)



In [7]:
dataset = pd.read_pickle('dog_prompts.pkl')
                         

In [8]:
for i, row in tqdm(dataset.iterrows(), total=len(dataset), desc="Generating Images"):
    prompt = row["Prompt"]
    image_path = os.path.join(output_dir, f"{i}.png")

    try:
        if os.path.isfile(image_path):
            with Image.open(image_path) as img:
                img.verify()
            print(f"Image {i} already exists and is valid. Skipping.")
            continue
    except (UnidentifiedImageError, OSError):
        print(f"Corrupted image at index {i}. Regenerating.")

    #print(f"Generating image {i} with prompt: {prompt}")
    generate_image_dalle3_hd(
        prompt=prompt,
        client=client,
        output_path=image_path
    )

Generating Images: 100%|██████████| 100/100 [27:37<00:00, 16.58s/it]
